# 01 — Exploratory Data Analysis & Preprocessing
### Single-cell RNA-seq: PBMC 3k (10x Genomics)

This notebook walks through loading raw counts, quality control, filtering,
normalization, and highly-variable-gene selection for the PBMC 3k dataset —
the same pipeline implemented in `src/data_processing.py`, shown here
step-by-step with visualizations so QC decisions are transparent and
reproducible.

**Why this matters:** in scRNA-seq, garbage-in-garbage-out is a bigger risk
than in most DL domains — empty droplets, dying cells, and doublets all
masquerade as real biological signal if not filtered carefully.

In [ ]:
import sys
sys.path.append("../src")

import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, facecolor="white")

## 1. Load raw data

Using the canonical PBMC 3k dataset (Zheng et al., 2017).

In [ ]:
from data_processing import load_raw_data, compute_qc_metrics

adata = load_raw_data()
adata = compute_qc_metrics(adata)
adata

## 2. QC metric distributions

Before choosing filtering thresholds, look at the actual distributions —
don't just copy thresholds from a tutorial blindly.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(adata.obs["n_genes_by_counts"], bins=60)
axes[0].set_title("Genes per cell")
axes[1].hist(adata.obs["total_counts"], bins=60)
axes[1].set_title("Total counts per cell")
axes[2].hist(adata.obs["pct_counts_mt"], bins=60)
axes[2].set_title("% mitochondrial counts")
plt.tight_layout()
plt.savefig("../results/qc_distributions.png", dpi=150)
plt.show()

## 3. Filter low-quality cells and genes

Thresholds (documented in `src/data_processing.py`):
- Keep cells with >= 200 genes detected (removes empty/near-empty droplets)
- Keep genes expressed in >= 3 cells (removes noise genes)
- Remove cells with > 15% mitochondrial reads (likely dying/stressed cells)
- Remove cells with > 4000 genes detected (likely doublets)

In [ ]:
from data_processing import filter_cells_and_genes

adata = filter_cells_and_genes(adata)
adata

## 4. Normalize (library size + log1p) and select highly variable genes

In [ ]:
from data_processing import normalize_data, select_highly_variable_genes

adata = normalize_data(adata)
adata = select_highly_variable_genes(adata, n_top_genes=2000)

sc.pl.highly_variable_genes(adata, save="_hvg.png")

## 5. Quick sanity-check visualization: PCA + UMAP on raw preprocessing

This is a *baseline* view before any deep learning — the VAE built in
notebook 02 should produce a latent space that separates cell types at
least as well as this, ideally better, especially on ambiguous/rare
populations.

In [ ]:
adata_hvg = adata[:, adata.var["highly_variable"]].copy()
sc.pp.scale(adata_hvg, max_value=10)
sc.tl.pca(adata_hvg, n_comps=50)
sc.pp.neighbors(adata_hvg)
sc.tl.umap(adata_hvg)
sc.tl.leiden(adata_hvg)

sc.pl.umap(adata_hvg, color=["leiden"], save="_baseline_pca_umap.png")

## 6. Save processed AnnData for the modeling stage

The full pipeline (equivalent to the cells above) is also available as a
single reusable script: `src/data_processing.py`.

In [ ]:
adata.write_h5ad("../data/pbmc3k_processed.h5ad")
print("Saved.")